In [1]:
from src.data.dataloader import get_dataloaders

train_loader, val_loader = get_dataloaders(data_dir="src/data/RRDataset_final", batch_size=32)

# Grab one batch
sample = next(iter(train_loader))

print(f"Images shape: {sample[0].shape}")       # Should be [32, 3, 224, 224]
print(f"Real/Fake labels: {sample[1]}")      # Should be a mix of 0s and 1s
print(f"Transform labels: {sample[2]}")   # Should be a mix of 0s, 1s, and 2s

Loaded 7200 images.
Images shape: torch.Size([32, 3, 224, 224])
Real/Fake labels: tensor([1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1,
        1, 0, 0, 0, 1, 0, 1, 0])
Transform labels: tensor([0, 2, 2, 0, 2, 2, 0, 1, 0, 2, 0, 1, 0, 2, 0, 2, 2, 1, 1, 2, 1, 0, 0, 2,
        2, 1, 0, 2, 1, 0, 1, 0])


In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Import your custom modules directly from the src directory
from src.train.loss import MultiTaskLoss
from src.train.loops import MultiTaskModel, train_epoch
from src.train.ablation import run_ablation_study

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# API Contract Requirements:
# - images: (B, 3, 224, 224) torch.float32
# - labels_real_fake: (B, 1) torch.float32
# - labels_transform: (B,) torch.long

B = 16 # Batch size for testing

dummy_images = torch.randn(B, 3, 224, 224)
dummy_labels_rf = torch.randint(0, 2, (B, 1)).float()
dummy_labels_tf = torch.randint(0, 3, (B,))

dataset = TensorDataset(dummy_images, dummy_labels_rf, dummy_labels_tf)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print("Mock dataset and dataloader initialized successfully.")

In [4]:

from torch.utils.data import DataLoader, TensorDataset

# --- 1. CREATE THE DATALOADER (This was the missing piece!) ---
B = 16 # Batch size for testing
dummy_images = torch.randn(B, 3, 224, 224)
dummy_labels_rf = torch.randint(0, 2, (B, 1)).float()
dummy_labels_tf = torch.randint(0, 3, (B,))

dataset = TensorDataset(dummy_images, dummy_labels_rf, dummy_labels_tf)
# Here is where 'dataloader' is officially defined:
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print("Dataloader initialized.")

# --- 2. INITIALIZE MODEL & TRAIN ---
model = MultiTaskModel().to(device)
criterion = MultiTaskLoss(alpha=0.5, beta=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Starting training test...")

# Run 1 training epoch
epoch_loss = train_epoch(model, dataloader, criterion, optimizer, device)

print(f"Test Epoch completed successfully! Average Loss: {epoch_loss:.4f}")

Dataloader initialized.
Starting training test...
Test Epoch completed successfully! Average Loss: 0.9105


In [5]:
print("Starting Ablation Study Runs...")
# This will execute the run_ablation_study() function from your module
run_ablation_study()

Starting Ablation Study Runs...
Running iteration with Alpha=0.5, Beta=0.5
Running iteration with Alpha=0.8, Beta=0.2
Running iteration with Alpha=0.2, Beta=0.8

Ablation Study Complete. Summary of losses:
Weights (alpha=0.5, beta=0.5) -> Average Loss: 0.7361
Weights (alpha=0.8, beta=0.2) -> Average Loss: 0.5622
Weights (alpha=0.2, beta=0.8) -> Average Loss: 0.8097
